# Solutions · 13. Regression trees and splitting by running sums

Worked solutions to the three exercises of [Lecture 13](../../notebooks/part5_ensembles/13_decision_trees_cart_and_prefix_sum_splitting.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: O(1) Prefix-Sum Split Discovery on a Synthetic Catalyst Light-Off Curve")
T_K = np.linspace(400.0, 560.0, 60)
# Synthetic light-off curve of a catalyst (logistic form, assumed constants) with a sharp ignition threshold at T_lightoff = 475 K:
X_conv = 0.05 + 0.92 / (1.0 + np.exp(-(T_K - 475.0) / 3.5))

split_1d = trees_ensembles.prefix_sum_best_split_1d(T_K, X_conv)
assert abs(split_1d["threshold"] - 475.0) < 3.0
print(f"True physical light-off T = 475.00 K | 1st Prefix-Sum CART Split = {split_1d['threshold']:.2f} K")
print(f"Total SST before split    = {split_1d['sse_reduction'] + split_1d['sse_after']:.4f} | SSE after 1st split = {split_1d['sse_after']:.4f} ({100*split_1d['sse_reduction']/(split_1d['sse_reduction']+split_1d['sse_after']):.1f}% variance explained by 1 split)")
heading("Demonstration 2: O(1) Prefix-Sum Identity vs. O(N^2) Brute-Force SSE Scan")
prefix_s = np.cumsum(X_conv)
s_tot = float(prefix_s[-1])
sst_conv = float(np.sum((X_conv - np.mean(X_conv)) ** 2))
k_idx = np.arange(1, len(X_conv))
s_left = prefix_s[:-1]
s_right = s_tot - s_left
gains_prefix = (s_left ** 2) / k_idx + (s_right ** 2) / (len(X_conv) - k_idx) - (s_tot ** 2) / len(X_conv)
sse_prefix = sst_conv - gains_prefix

sse_brute = np.array([
    float(np.sum((X_conv[:k] - np.mean(X_conv[:k])) ** 2) + np.sum((X_conv[k:] - np.mean(X_conv[k:])) ** 2))
    for k in k_idx
])
max_diff_sse = float(np.max(np.abs(sse_prefix - sse_brute)))
print(f"Max discrepancy across all 59 candidate splits ||SSE_prefix - SSE_brute||_max: {max_diff_sse:.2e}")
assert max_diff_sse < 1e-13
heading("Demonstration 3: Recursive CART Tree Depth Progression & Exact Mean Conservation")
sse_by_depth = []
print(f"{'Max Depth D':>11s} | {'Max Leaves 2^D':>14s} | {'Residual SSE':>14s} | {'Mean Conservation |sum(y_hat - y)|':>34s}")
print("-" * 82)
for depth in [1, 2, 3, 4]:
    tree = trees_ensembles.DecisionTreeRegressor(max_depth=depth, min_samples_split=2).fit(T_K, X_conv)
    pred = tree.predict(T_K)
    sse_val = float(np.sum((pred - X_conv) ** 2))
    mean_err = abs(float(np.sum(pred) - np.sum(X_conv)))
    sse_by_depth.append(sse_val)
    print(f"{depth:11d} | {2**depth:14d} | {sse_val:14.6f} | {mean_err:34.2e}")
    assert np.isclose(np.sum(pred), np.sum(X_conv), rtol=1e-13)

assert all(sse_by_depth[i] > sse_by_depth[i + 1] for i in range(len(sse_by_depth) - 1))
heading("Demonstration 4: Constant Prediction of a Tree Outside the Training Range (Arrhenius Rate)")
T_train = np.linspace(300.0, 400.0, 30)
k_train = 1e6 * np.exp(-5000.0 / T_train)
tree_arr = trees_ensembles.DecisionTreeRegressor(max_depth=4, min_samples_split=2).fit(T_train, k_train)

T_extrap = np.array([400.0, 450.0, 500.0, 600.0])
k_true_extrap = 1e6 * np.exp(-5000.0 / T_extrap)
k_pred_extrap = tree_arr.predict(T_extrap)
print(f"{'Temperature [K]':>15s} | {'True Arrhenius Rate':>19s} | {'CART Tree Prediction':>20s} | {'Relative Error':>14s}")
print("-" * 76)
for t_v, kt, kp in zip(T_extrap, k_true_extrap, k_pred_extrap):
    print(f"{t_v:15.1f} | {kt:19.4f} | {kp:20.4f} | {100*abs(kp-kt)/kt:13.1f}%")
assert np.allclose(k_pred_extrap, k_pred_extrap[0], atol=1e-15)
heading("Inside the Algorithm: By-Hand Prefix-Sum Best Split vs. the Library")
best_i = int(np.argmax(gains_prefix))
th_hand = 0.5 * float(T_K[best_i] + T_K[best_i + 1])
gain_hand = float(gains_prefix[best_i])
print(f"by hand (pure NumPy): threshold = {th_hand:.6f} K, gain = {gain_hand:.6f}")
print(f"library (engml_2_applied)     : threshold = {split_1d['threshold']:.6f} K, gain = {split_1d['sse_reduction']:.6f}")
assert np.isclose(th_hand, split_1d["threshold"], rtol=1e-14) and np.isclose(gain_hand, split_1d["sse_reduction"], rtol=1e-14)
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify mathematically and numerically on a 40-point polymer step-change dataset $C_p(T)$ with a glass-transition jump at $T_g = 373\;\mathrm{K}$ that `prefix_sum_best_split_1d` produces the exact same optimal threshold and post-split SSE as a brute-force loop evaluating $\sum_{i \in L}(y_i - \bar{y}_L)^2 + \sum_{i \in R}(y_i - \bar{y}_R)^2$ at every midpoint.

In [2]:
heading("Exercise 1: Polymer Glass-Transition (T_g = 373 K) Step Discovery")
T_poly = np.linspace(320.0, 420.0, 40)
y_tg = np.where(T_poly <= 373.0, 0.85, 0.98)
ps_res = trees_ensembles.prefix_sum_best_split_1d(T_poly, y_tg)

mids = 0.5 * (T_poly[:-1] + T_poly[1:])
bf_sses = [
    float(np.sum((y_tg[T_poly <= m] - np.mean(y_tg[T_poly <= m])) ** 2) + np.sum((y_tg[T_poly > m] - np.mean(y_tg[T_poly > m])) ** 2))
    for m in mids
]
best_idx = int(np.argmin(bf_sses))
assert np.isclose(ps_res["threshold"], mids[best_idx], rtol=1e-14)
assert np.isclose(ps_res["sse_after"], bf_sses[best_idx], atol=1e-13)
print(f"Prefix-sum threshold = {ps_res['threshold']:.2f} K == Brute-force threshold = {mids[best_idx]:.2f} K (Post-split SSE = {ps_res['sse_after']:.2e})")


Exercise 1: Polymer Glass-Transition (T_g = 373 K) Step Discovery
Prefix-sum threshold = 372.56 K == Brute-force threshold = 372.56 K (Post-split SSE = 2.19e-15)


**Discussion.** For a clean two-regime step function at $T_g = 373\;\mathrm{K}$, a single depth-1 CART split reduces the post-split SSE to $0.0000$, locating the exact midpoint containing $T_g$.

---

## Exercise 2

**Problem.** Audit the **high-variance overfitting** of an unregularized deep decision tree (`max_depth=12, min_samples_split=2`) vs. a regularized shallow tree (`max_depth=2`) on noisy glass-transition data (`y_tg_noisy = y_tg + N(0, 0.04^2)`): show that the deep tree memorizes the training noise ($\mathrm{SSE}_{\mathrm{train}} \approx 0$) but suffers $>2\times$ higher test MSE against the true step curve `y_tg`.

In [3]:
heading("Exercise 2: Unpruned Deep Tree Overfitting Audit on Noisy T_g Data")
rng_tg = np.random.default_rng(42)
y_tg_noisy = y_tg + rng_tg.normal(0.0, 0.04, size=40)
tree_shallow = trees_ensembles.DecisionTreeRegressor(max_depth=1, min_samples_split=2).fit(T_poly, y_tg_noisy)
tree_deep = trees_ensembles.DecisionTreeRegressor(max_depth=12, min_samples_split=2).fit(T_poly, y_tg_noisy)

mse_tr_deep = float(np.mean((tree_deep.predict(T_poly) - y_tg_noisy) ** 2))
mse_true_shallow = float(np.mean((tree_shallow.predict(T_poly) - y_tg) ** 2))
mse_true_deep = float(np.mean((tree_deep.predict(T_poly) - y_tg) ** 2))
print(f"Deep tree (depth=12) training MSE vs noisy labels : {mse_tr_deep:.2e} (Memorized noise)")
print(f"MSE against true physical T_g curve: Shallow (D=1) = {mse_true_shallow:.6f} vs Deep (D=12) = {mse_true_deep:.6f}")
assert mse_tr_deep < 1e-20 and mse_true_deep > 5.0 * mse_true_shallow


Exercise 2: Unpruned Deep Tree Overfitting Audit on Noisy T_g Data
Deep tree (depth=12) training MSE vs noisy labels : 0.00e+00 (Memorized noise)
MSE against true physical T_g curve: Shallow (D=1) = 0.000013 vs Deep (D=12) = 0.001066


**Discussion.** An unpruned tree with `min_samples_split=2` splits until every single noisy measurement occupies its own 1-sample leaf, and reproduces the noise: its error against the true curve is $1.1\times 10^{-3}$, of the order of the noise variance $0.04^2 = 1.6\times 10^{-3}$ and about 80 times that of the tree with one split ($1.3\times 10^{-5}$).

---

## Exercise 3

**Problem.** Implement a **cost-complexity pruned depth selector** `select_tree_depth_bic(x: np.ndarray, y: np.ndarray, max_depths: list[int], alpha_pen: float = 0.005) -> dict` that evaluates Breiman's cost-complexity objective $R_\alpha(T_D) = \mathrm{SSE}(T_D) + \alpha_{\mathrm{pen}} |T_D|$ (where $|T_D|$ is the number of distinct leaf values) and verify on `(T_poly, y_tg_noisy)` that it selects the true 2-regime step tree $D^* = 1$.

In [4]:
heading("Exercise 3: Implement Breiman Cost-Complexity Tree Pruning Selector")
def count_leaves(node: dict) -> int:
    if node["leaf"]:
        return 1
    return count_leaves(node["left"]) + count_leaves(node["right"])

def select_tree_depth_cc(x: np.ndarray, y: np.ndarray, max_depths: list[int], alpha_pen: float = 0.005) -> dict:
    """Select CART tree depth minimizing Breiman's cost-complexity R_alpha(T) = SSE(T) + alpha * |T|."""
    scores = {}
    for d_val in max_depths:
        t_m = trees_ensembles.DecisionTreeRegressor(max_depth=d_val, min_samples_split=2).fit(x, y)
        sse_m = float(np.sum((t_m.predict(x) - y) ** 2))
        n_leaves = count_leaves(t_m.tree_)
        scores[d_val] = sse_m + alpha_pen * n_leaves
        print(f"Depth D = {d_val:2d} | Leaves |T| = {n_leaves:2d} | SSE = {sse_m:.5f} | R_alpha(T) = {scores[d_val]:.5f}")
    best_d = min(scores, key=scores.get)
    return {"best_depth": best_d, "scores": scores}

cc_res = select_tree_depth_cc(T_poly, y_tg_noisy, [1, 2, 3, 5, 8], alpha_pen=0.005)
print(f"Cost-complexity selected tree depth D* = {cc_res['best_depth']}")
assert cc_res["best_depth"] == 1


Exercise 3: Implement Breiman Cost-Complexity Tree Pruning Selector
Depth D =  1 | Leaves |T| =  2 | SSE = 0.04212 | R_alpha(T) = 0.05212
Depth D =  2 | Leaves |T| =  4 | SSE = 0.03776 | R_alpha(T) = 0.05776
Depth D =  3 | Leaves |T| =  8 | SSE = 0.02427 | R_alpha(T) = 0.06427
Depth D =  5 | Leaves |T| = 27 | SSE = 0.00390 | R_alpha(T) = 0.13890
Depth D =  8 | Leaves |T| = 39 | SSE = 0.00000 | R_alpha(T) = 0.19500
Cost-complexity selected tree depth D* = 1


**Discussion.** Penalizing leaf count via Breiman's cost-complexity criterion $R_\alpha(T) = \mathrm{SSE}(T) + \alpha |T|$ keeps a tree from dividing a plateau into leaves of single noisy points.